In [ ]:
REPO_URL = "https://github.com/30nikamvaishnavi/autonomous-driving-perception-stack.git"
BRANCH = "main"
DATA_DIR = None  # Override to use a different runtime dataset path.
OUT_DIR = None  # Override to change where outputs and checkpoints are saved.
WANDB_PROJECT = "adpstack"
MODEL_PATH = None  # Optional .pt path for export/benchmark notebooks.


In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

SMOKE_STARTED_AT = time.monotonic()
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = Path("/kaggle").exists()
if IS_COLAB == IS_KAGGLE:
    raise RuntimeError("Unsupported or ambiguous runtime: use Google Colab or Kaggle.")

if IS_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    runtime_root = Path("/content")
    data_default = Path("/content/drive/MyDrive/adpstack/data/bdd100k")
    output_default = Path("/content/drive/MyDrive/adpstack/outputs")
else:
    runtime_root = Path("/kaggle/working")
    data_default = Path("/kaggle/input")
    output_default = Path("/kaggle/working/outputs")

DATA_DIR = Path(DATA_DIR) if DATA_DIR is not None else data_default
OUT_DIR = Path(OUT_DIR) if OUT_DIR is not None else output_default
OUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ["WANDB_PROJECT"] = WANDB_PROJECT

REPO_DIR = runtime_root / "autonomous-driving-perception-stack"
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
elif REPO_DIR.exists():
    raise FileExistsError(f"{REPO_DIR} exists but is not a Git checkout; remove/rename it and retry.")
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[ml,export]"],
    cwd=REPO_DIR,
    check=True,
)
import torch

print(f"Runtime: {'Colab' if IS_COLAB else 'Kaggle'}")
print(f"Repository: {REPO_DIR}")
print(f"Data: {DATA_DIR}")
print(f"Outputs: {OUT_DIR}")
print(f"PyTorch: {torch.__version__}")
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is attached. Select a T4 GPU runtime in Colab or enable a GPU accelerator in Kaggle, then restart."
    )
print(f"GPU: {torch.cuda.get_device_name(0)}")


# 00 - GPU smoke test

Run on a Colab T4 or Kaggle GPU. The tiny synthetic dataset and single YOLO11n epoch are only a pipeline check, not a benchmark.

## Install the lightweight test runner and run the repository suite

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO_DIR, check=True)

## Convert 20 synthetic BDD-format frames

In [ ]:
import json
import tempfile

import cv2
import numpy as np
from adpstack.data.bdd100k_to_yolo import convert_split, write_dataset_yaml

SMOKE_TMP = tempfile.TemporaryDirectory(prefix="adpstack-bdd-smoke-")
SMOKE_ROOT = Path(SMOKE_TMP.name)
SMOKE_DATA_DIR = SMOKE_ROOT / "yolo"
SMOKE_IMAGE = None
frame_index = 0
for split, count in (("train", 16), ("val", 4)):
    image_dir = SMOKE_ROOT / "source" / "images" / split
    labels_dir = SMOKE_ROOT / "source" / "labels" / "det_20"
    image_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)
    frames = []
    for _ in range(count):
        name = f"frame_{frame_index:02d}.jpg"
        image = np.zeros((72, 128, 3), dtype=np.uint8)
        cv2.rectangle(image, (32, 18), (96, 54), (0, 255, 0), -1)
        image_path = image_dir / name
        if not cv2.imwrite(str(image_path), image):
            raise OSError(f"Failed to write synthetic image: {image_path}")
        frames.append({
            "name": name,
            "labels": [{"category": "car", "box2d": {"x1": 320, "y1": 180, "x2": 960, "y2": 540}}],
        })
        if SMOKE_IMAGE is None:
            SMOKE_IMAGE = image_path
        frame_index += 1
    labels_path = labels_dir / f"det_{split}.json"
    labels_path.write_text(json.dumps(frames), encoding="utf-8")
    converted = convert_split(labels_path, image_dir, SMOKE_DATA_DIR, split, copy_images=True)
    assert converted == count, f"Expected {count} converted {split} frames, got {converted}"

SMOKE_DATASET = write_dataset_yaml(SMOKE_DATA_DIR)
print(f"Converted {frame_index} synthetic frames: {SMOKE_DATASET}")

## Train YOLO11n for one epoch on the attached GPU

In [ ]:
from ultralytics import YOLO, settings

settings.update({"wandb": False})
SMOKE_RUN_DIR = OUT_DIR / "gpu_smoke"
model = YOLO("yolo11n.pt")
model.train(
    data=str(SMOKE_DATASET),
    epochs=1,
    imgsz=64,
    batch=4,
    device=0,
    workers=2,
    project=str(SMOKE_RUN_DIR),
    name="yolo11n",
    exist_ok=True,
    plots=False,
    verbose=False,
    cache=False,
    patience=1,
)
SMOKE_BEST = SMOKE_RUN_DIR / "yolo11n" / "weights" / "best.pt"
if not SMOKE_BEST.is_file():
    raise FileNotFoundError(f"Smoke-test checkpoint not found: {SMOKE_BEST}")
print(f"Checkpoint: {SMOKE_BEST}")

## Export ONNX and exercise the project's ONNX decoder

In [ ]:
from adpstack.detection.infer import Detector

smoke_model = YOLO(str(SMOKE_BEST))
SMOKE_ONNX = Path(smoke_model.export(format="onnx", imgsz=64, opset=17, simplify=False))
if not SMOKE_ONNX.is_file():
    raise FileNotFoundError(f"ONNX export not found: {SMOKE_ONNX}")
detector = Detector(str(SMOKE_ONNX), imgsz=64, conf=0.01)
image = cv2.imread(str(SMOKE_IMAGE))
if image is None:
    raise OSError(f"Failed to read synthetic image: {SMOKE_IMAGE}")
detections = detector(image)
assert detections.xyxy.ndim == 2 and detections.xyxy.shape[1] == 4
elapsed = time.monotonic() - SMOKE_STARTED_AT
print(f"ONNX decoder completed ({len(detections)} detections); elapsed: {elapsed:.1f}s")
if elapsed > 600:
    raise TimeoutError(f"GPU smoke test took {elapsed / 60:.1f} minutes; target is under 10 minutes.")